# 🔭 Pydantic Logfire — Production LLM Observability

**Instructor: Divesh | Production-Grade LLM Engineering**

---

## What You'll Build

In this notebook we go from **zero observability** to a **fully traced RAG pipeline** step by step.

| Part | What We Cover |
|------|--------------|
| **Part 1** | Why Logfire, first spans, structured logging |
| **Part 2** | Auto-instrument Groq + Gemini — tokens, latency, model name for free |
| **Part 3** | Full RAG tracing — embed → retrieve → generate waterfall |

**Stack:**
- `Logfire` — Python-native observability (built on OpenTelemetry)
- `Groq (llama-3.3-70b)` + `Gemini (gemini-2.5-flash-lite)` — two providers, one dashboard
- `FAISS` + `Gemini text-embedding (gemini-embedding-2-preview)` — API-based vector search
- `LangChain` + `LangGraph` — RAG orchestration and ReAct agent

> 📌 **Keep your Logfire dashboard open in a browser tab** — you'll see events appear in real time as each cell runs.

## SETUP LOGFIRE

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv()


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GOOGLE_API_KEY :", "✅" if os.getenv("GOOGLE_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GOOGLE_API_KEY : ✅


---
## 🧱 Part 1 — Why Logfire & First Traces

The problem with `print()` in production:

| `print()` | `logfire` |
|-----------|-----------|
| Plain string, unsearchable | Structured key-value fields, fully searchable |
| No timestamp or duration | Automatic timestamps, span duration |
| Lost in terminal noise | Real-time dashboard with filters and queries |
| Nothing in production | Persisted traces, alerting, cost analytics |

Logfire is built on **OpenTelemetry** — the industry standard. Every trace you write here is portable.

### 🧪 Experiment 1 — Configure Logfire & First Manual Span

**Three primitives to learn:**

1. `logfire.configure()` — one-time setup, connects to your project dashboard
2. `logfire.info(msg, **attrs)` — a structured log event (NOT a string, it's a searchable record)
3. `with logfire.span("name", **attrs):` — a *timed block* with a name and attributes

After running this cell → switch to your **Logfire dashboard** and watch these events appear.

In [2]:
import logfire
logfire.configure()
logfire.info('hello, {place}!', place = "world")

Logfire project URL: https://logfire-us.pydantic.dev/jdevaiya/first-project

17:18:53.599 hello, world!


In [3]:
logfire.configure(
    token=os.getenv("LOGFIRE_TOKEN"),
    service_name="llm-observability-course"
)

Logfire project URL: https://logfire-us.pydantic.dev/jdevaiya/first-project

## SIMPLE SPAN

In [4]:
logfire.info("notebook_start",
            part="PART 1 - BASICS",
            instructor = "Jayendra",
            tool = "Pydentic Logfire")

17:23:56.311 notebook_start


TRACE

In [5]:
with logfire.span("data_processing_simulation",dataset = "llm_course",rows=1000):
    logfire.info("step_started",step=1,action="loading data")
    time.sleep(0.3)
    
    logfire.info("step_started", step=2,action = "transforming",columns=12)
    time.sleep(0.2)
    
    logfire.info("step_started",step=3,action="saving_result",output = "/tmp/out.csv")

17:30:06.585 data_processing_simulation
17:30:06.599   step_started
17:30:06.901   step_started
17:30:07.103   step_started


### 🧪 Experiment 2 — Structured Logging with Pydantic Models

The *"Pydantic"* in Pydantic Logfire: when you log a Pydantic model, Logfire **expands every field** into a searchable attribute automatically.

In a real LLM app you log request/response objects hundreds of times per minute. With string logging you get `"{'user_id': 'alice', ...}"` — unsearchable. With Logfire you get filterable columns.

In [8]:
from pydantic import BaseModel
from typing import Optional

# mock data not real data

class LLMRequest(BaseModel):
    user_id:str
    session_id:str
    query:str
    model:str
    temperature:float= 0.7
    max_tokens:Optional[int] = None
    
class LLMResponse(BaseModel):
    answer :str
    input_tokens:int
    output_tokens:int
    latency_ms:float
    model_used:str

PASS DATA

In [9]:
# ── Simulate logging a real LLM request/response ──────────────────────────

request = LLMRequest(
    user_id="priya",
    session_id="sess_abc123",
    query="what is Retrieval-Ahgmented Generation?",
    model = "groq:openai/gpt-oss-120b",
    max_tokens=500
)

with logfire.span("llm_CALL",
                user_id = request.user_id,
                session_id= request.session_id,
                model_used = request.model):
    logfire.info(
        "request_received",
        **request.model_dump()
    )
    
    time.sleep(0.1)
    
    
    response = LLMResponse(
            answer = "RAG is a technique that retrieves relevant documents...",
            input_tokens = 18,
            output_tokens=120,
            latency_ms=342.5,
            model_used="groq:openai/gpt-oss-120b"
    )
    
    logfire.info("response_sent", **response.model_dump())
    
    print(response)

17:47:44.295 llm_CALL
17:47:44.317   request_received
17:47:44.422   response_sent
answer='RAG is a technique that retrieves relevant documents...' input_tokens=18 output_tokens=120 latency_ms=342.5 model_used='groq:openai/gpt-oss-120b'


---
## ⚡ Part 2 — Auto-Instrumentation of LLM Calls

In Part 1 you wrote spans manually. Now let Logfire do it **automatically**.

`logfire.instrument_openai()` patches the OpenAI Python SDK.
Since **Groq** and **Gemini** both expose an OpenAI-compatible REST API, the same single line instruments all of them.

Every LLM call then automatically records:
- Model name (`llama-3.3-70b-versatile`, `gemini-2.5-flash-lite`, …)
- Input + output token counts
- Wall-clock latency
- Full prompt text and full response text

You write **zero** extra logging code. It just appears in the dashboard.

### 🧪 Experiment 3 — Instrument Groq (llama-3.3-70b)

We use `ChatOpenAI` from LangChain pointed at Groq's OpenAI-compatible endpoint.
`logfire.instrument_openai()` patches the underlying SDK — Groq calls appear in traces automatically.

```
Your code  →  ChatOpenAI(base_url="https://api.groq.com/openai/v1")
                          ↓
              [logfire.instrument_openai() intercepts here]
                          ↓
              Groq API  →  response
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model = "openai/gpt-oss-120b",
    temperature=0.3
)

# make a call - watch the trace appear in the dashboard automatically

print("calling Groq(openai/gpt-oss-120b)")
response = llm_groq.invoke([
    HumanMessage(
        content="Explain what an observability 'span' is, in exactly 2 sentences"
    )
])

print(response.content)

calling Groq(groq:openai/gpt-oss-120b)
17:55:02.492 Chat Completion with 'openai/gpt-oss-120b' [LLM]
An observability *span* is a timed unit of work representing a single operation or segment of a distributed trace, capturing its start and end timestamps along with metadata such as attributes, events, and status. Spans are linked together to form a trace, allowing engineers to visualize and analyze the end‑to‑end flow of requests across services and pinpoint latency or failure points.


### 🧪 Experiment 4 — Instrument Gemini (gemini-2.5-flash-lite)

Gemini now exposes an OpenAI-compatible endpoint too:
`https://generativelanguage.googleapis.com/v1beta/openai/`

Same `ChatOpenAI` setup — just a different `base_url` and model name.
`logfire.instrument_openai()` was already called — no need to call it again.

After this cell, your dashboard will show traces from **two different model names** in the same project.

In [18]:
# Gemini's OpenAI-compatible endpoint (no extra setup needed)
llm_gemini = ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY"), #type:ignore
    model="gemini-3.7-flash",
    temperature=0.3
)

print("calling Gemini (gemini-3.8-flash)")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Explain what an observability 'trace' is, in exactly 2 sentences.")
        
    ])
    
    print(f"\n🔵 gemini Response:\n{response.content}")
    
except Exception as e:
    print(f"⚠️ Gemini call failed: {e}")
    print("Check your GOOGLE_API_KEY in .env")

calling Gemini (gemini-3.8-flash)
18:21:36.200 Chat Completion with 'gemini-3.7-flash' [LLM]
⚠️ Gemini call failed: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Check your GOOGLE_API_KEY in .env


### 🧪 Experiment 5 — Side-by-Side: Groq vs Gemini in One Trace Waterfall

Wrap both calls in a **parent span** — the dashboard shows them as a waterfall:

```
model_comparison  ←── parent span (total wall time)
  ├── groq_call   ←── child span  (latency: Xms, tokens: N)
  └── gemini_call ←── child span  (latency: Xms, tokens: N)
```

This is exactly how you run **A/B model tests in production** — one trace per comparison, all searchable.

In [34]:
query = "What is the difference between RAG and fine-tuning? Give 3 bullet points."

with logfire.span("model_comparison",query=query,num_models=2):
    # ── Groq ─────────────────────────────────────────────────────────────
    
    with logfire.span("groq_call",model="openai/gpt-oss-20b",provider ="groq"):
        t0= time.time()
        r_groq= llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time()-t0)*1000,1)
        logfire.info("groq_done",latency_ms=groq_ms)
        answer_len=len(r_groq.content)
        
        
    # ── Gemini ────────────────────────────────────────────────────────────
    with logfire.span("gemini_call",model="gemini-3.7-flash",provider ="google"):
            t0= time.time()
            try:
                r_gemini= llm_groq.invoke([HumanMessage(content=query)])
                gemini_ms = round((time.time()-t0)*1000,1)
                logfire.info("gemini_done",latency_ms=gemini_ms)
                answer_len=len(r_gemini.content)
                gemini_answer = r_gemini.content
            except Exception as e:
                logfire.warning("gemini_failed", error=str(e))
                gemini_ms = 0
                gemini_answer = f"[Error: {e}]"
    
    
# ── Print results ─────────────────────────────────────────────────────────

print(f"🟢 Groq ({groq_ms} ms) : \n {r_groq.content}")
print(f"\n🔵 Gemini ({gemini_ms} ms) : \n {r_gemini.content}") # type: ignore

19:47:12.269 model_comparison
19:47:12.270   groq_call
19:47:12.274     Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:13.314     groq_done
19:47:13.315   gemini_call
19:47:13.318     Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:14.724     gemini_done
🟢 Groq (1042.8 ms) : 
 - **How knowledge is incorporated**  
  - **RAG:** Retrieves relevant external documents at inference time and feeds them to the language model, so the model can use up‑to‑date information without changing its parameters.  
  - **Fine‑tuning:** Embeds new knowledge directly into the model’s weights by training on additional data, making the information permanent but static until the next training cycle.

- **Resource and latency trade‑offs**  
  - **RAG:** Requires a searchable index and a retrieval step for each query, which adds latency but avoids the heavy compute cost of re‑training.  
  - **Fine‑tuning:** Involves a one‑off compute‑intensive training run; after that, inference is fast because 

---
## 📚 Part 3 — RAG Pipeline Tracing

A RAG pipeline has **3 stages**, each taking time and each capable of failing:

```
User Query
    │
    ▼
[Embed Query]        ← Gemini text-embedding API call
    │
    ▼
[Retrieve Docs]      ← FAISS similarity search (in-memory)
    │
    ▼
[Generate Answer]    ← LLM API call (Groq llama-3.3-70b)
    │
    ▼
Response
```

Without observability: *"It's slow"* — but which stage? The embedding? The retrieval? The LLM?
With Logfire: you see **exactly** which stage takes how long and what data flows through each one.

**Stack:**
- Embeddings: `gemini-embedding-2-preview` via Gemini API — no local model, reuses your existing key
- Vector store: `FAISS` — in-memory, no server needed
- LLM: Groq `llama-3.3-70b-versatile`

### 🧪 Experiment 6 — Traced RAG Pipeline

Load documents from `documents.json`, embed them with Gemini, build a FAISS index, and run a traced RAG pipeline.

Tracing shows the two things that matter:
- **Which documents were retrieved** (relevance check without opening the code)
- **LLM call** — model, tokens, latency, full prompt+response (auto-captured by `instrument_openai`)

In [35]:
import json 
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# ── Load knowledge base ────────────────────────────────────────────────────

with open("documents.json")as f:
    raw_docs = json.load(f)
    
# CONVERT THE DOCS IN LANGCHAIN COMPATIPBLE DOCUMENTS
DOCS =[
    Document(page_content=d["content"],metadata = {'topic':d['topic'],"source":d["source"]})
    for d in raw_docs
]

print(f"Loaded {len(DOCS)} documents: {[d.metadata['topic']for d in DOCS]}")

Loaded 6 documents: ['RAG', 'Guardrails', 'Gateway', 'Observability', 'Evals', 'Fine-tuning']


In [36]:
DOCS

[Document(metadata={'topic': 'RAG', 'source': 'doc_1'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. When a user asks a question, RAG first retrieves relevant documents from a knowledge base using vector similarity search, then passes those documents along with the question to an LLM. This grounds the answer in actual content, which significantly reduces hallucinations compared to pure LLM generation.'),
 Document(metadata={'topic': 'Guardrails', 'source': 'doc_2'}, page_content='LLM Guardrails are safety controls that sit between the user and the language model. They run before the LLM sees the input (input rails) and after the LLM generates output (output rails). NVIDIA NeMo Guardrails uses a domain-specific language called Colang to define rules declaratively. Common guardrails include prompt injection detection, PII filtering, toxicity filtering, and topic restriction.'),
 Document(metadata={'topic': 'Gateway', 'source': 'd

In [37]:
# ── Embeddings + FAISS index ───────────────────────────────────────────────
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY") #type:ignore
    
)

vectorstore = FAISS.from_documents(DOCS,embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
print("✅ FAISS index ready")

✅ FAISS index ready


DATA INGESTION DONE

DATA RETRIEVAL PIPELINE

In [38]:
# ── RAG with tracing ───────────────────────────────────────────────────────

def rag(question:str,user_id:str ="anonymous") -> str :
    with logfire.span("rag_pipeline", question=question,user_id=user_id):
        docs = retriever.invoke(question) #search similar vectors
        
        logfire.info("docs_retrieved",
                    topics=[d.metadata["topic"] for d in docs],
                    num_docs=len(docs))
        
        context = "\n\n".join(
            f"[{d.metadata["topic"]}] {d.page_content}"
            for d in docs
        )
        
        prompt = (
            f"Answer the question based only on the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer concisely:"
        )
        
        return llm_groq.invoke(prompt).content #type:ignore

In [39]:
answer = rag("How does a Rag reduce hallucination",user_id="student1")
print(f"\nA:{answer}")

19:47:29.690 rag_pipeline
19:47:30.332   docs_retrieved
19:47:30.335   Chat Completion with 'openai/gpt-oss-120b' [LLM]

A:RAG reduces hallucination by first retrieving relevant documents from a knowledge base and feeding those concrete sources to the LLM along with the user’s question. Because the model’s answer is grounded in retrieved, factual content (rather than generated solely from its internal weights), it is less likely to fabricate information.


REACT AGENT

### 🧪 Experiment 7 — LangGraph ReAct Agent with Retriever as a Tool

A fixed RAG chain always retrieves. An agent decides *when* to retrieve.

The retriever is a plain Python function decorated with `@tool` — the agent calls it only when the question needs knowledge-base lookup.

```
agent_run  (logfire span)
  ├── LLM decides: call tool or answer directly?
  ├── search_knowledge_base()  ← only if needed
  └── LLM generates final answer
```

In [40]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage,AIMessage

In [41]:
# ── Retriever as a plain Python tool ──────────────────────────────────────
@tool
def search_knowledge_base(query:str)->str:
    """ Search the knowledge base for LLM production topics: RAG, Guardrails, Gateways, observability, evaluations, and fine-tuning. """
    docs = vectorstore.similarity_search(query,k=2)
    return "\n\n".join(
        f"[{d.metadata['topic']}] {d.page_content}" for d in docs
    )

In [42]:
agent= create_agent(
    model = llm_groq,
    tools = [search_knowledge_base],
    system_prompt=(
        "You are a helpful assistant. Use search_knowledge_base for any question "
                "about LLM production topics. Answer directly for general knowledge questions."
    )
)

In [43]:
# ── Traced agent runner ────────────────────────────────────────────────────

def run_agent(question: str, user_id: str = "anonymous"):
    with logfire.span("agent_run", question=question, user_id=user_id):
        result = agent.invoke({"messages": [HumanMessage(content=question)]})

        # Last message may be a ToolMessage or an AIMessage with empty content
        # — iterate backwards to find the last AIMessage with actual text
        last_ai = next(
            (m for m in reversed(result["messages"]) if isinstance(m, AIMessage)),
            None
        )
        answer    = last_ai.text if last_ai else ""
        used_tool = any(isinstance(m, ToolMessage) for m in result["messages"])

        logfire.info("agent_done", used_tool=used_tool, answer_length=len(answer))
        return answer, used_tool


In [44]:
# ── Test ───────────────────────────────────────────────────────────────────
queries = [
    ("What is LLM observability and which tools provide it?", "priya"),
    ("How do LLM guardrails work?",  "bhavesh"),
    ("What is the capital of France?", "kunal"),
]

In [45]:
for q, uid in queries:
    print(f"{'='*55}")
    answer, used_tool = run_agent(q, user_id=uid)
    print(f"Q: {q}")
    print(f"Tool used: {used_tool}  {'← retrieved from KB' if used_tool else '← answered directly'}")
    print(f"A: {answer[:300]}")

19:47:34.286 agent_run
19:47:34.292   Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:35.583   Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:37.016   agent_done
Q: What is LLM observability and which tools provide it?
Tool used: True  ← retrieved from KB
A: **LLM observability** is the ability to monitor, trace, and debug language‑model‑driven applications once they are running in production.  
A good observability stack for LLMs typically captures:

| Aspect | What it means for LLMs |
|--------|------------------------|
| **Token & cost tracking** | R
19:47:37.017 agent_run
19:47:37.021   Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:38.145   Chat Completion with 'openai/gpt-oss-120b' [LLM]
19:47:42.644   agent_done
Q: How do LLM guardrails work?
Tool used: True  ← retrieved from KB
A: **LLM guardrails** are safety‑and‑policy controls that sit between a user (or upstream system) and a large language model (LLM). Their purpose is to keep the model’s behavior wi